# Introduction

In this notebook, we use the OpenClip model as transformer and trainform all the images into vectors, then feed the vectors to multiple ML algorithms.

## Convert Images into DataFrame

In [ ]:
### Import Libraries

In [1]:
import torch
import os
import joblib

from PIL import Image
import open_clip
import numpy as np
import pandas as pd

from nazi_symbols_classification.training.data_preparation import get_image_paths
from nazi_symbols_classification.training.evaluation import get_top1_evaluation
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, roc_curve, auc, precision_recall_curve, average_precision_score

/mnt/data/nazi-symbols-classification/venv/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
model.eval()  # model in train mode by default, impacts some models with BatchNorm or stochastic depth active

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-11): 12 x ResidualAttentionBlock(
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=768, out_features=3072, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((768,), eps=1e-05, elementwise_affine

### Get image paths

In [3]:
dir_name = os.path.dirname(os.getcwd())
images = get_image_paths(f"{dir_name}/datasets/nazi-symbols-detection", 
                         ("train", "test", "val"))

In [4]:
train_images = [image for image in images if image.startswith(f'{dir_name}/datasets/nazi-symbols-detection/train')]
test_images = [image for image in images if image.startswith(f'{dir_name}/datasets/nazi-symbols-detection/test')]
valid_images = [image for image in images if image.startswith(f'{dir_name}/datasets/nazi-symbols-detection/val')]

In [5]:
len(train_images), len(test_images), len(valid_images)

(75369, 15018, 15376)

In [6]:
y_train = [os.path.basename(os.path.dirname(image)) for image in train_images]
y_test = [os.path.basename(os.path.dirname(image)) for image in test_images]
y_valid = [os.path.basename(os.path.dirname(image)) for image in valid_images]

In [7]:
y_train.count("nazi-symbol"), y_test.count("nazi-symbol"), y_valid.count("nazi-symbol")

(4038, 205, 403)

## Preprocess images and save to CSV

In [8]:
def load_image(image_path):
    with torch.no_grad(), torch.autocast("cuda"):
        image = preprocess(Image.open(image_path)).unsqueeze(0)
        image_features = model.encode_image(image)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        return pd.DataFrame(image_features.float().numpy())

def preprocess_images(images):
    return pd.concat([load_image(image_path) for image_path in images], axis=0)

In [9]:
load_image(train_images[1])

,0,1,2,3,4,5,6,7,8,9,...,502,503,504,505,506,507,508,509,510,511
0,-0.015738,0.093096,0.084752,0.112216,-0.000442,-0.065532,0.007091,-0.025539,-0.04525,0.040717,...,0.05709,-0.064796,-0.022393,0.011908,-0.00822,-0.018699,0.013882,0.019758,-0.008252,-0.018672


In [ ]:
with open("training_data.csv", "w") as f:
    f.write(",".join(map(str, range(512))))
    f.write("\n")

for i in range(0, len(train_images), 200):
    data = preprocess_images(train_images[i:i+200])
    with open("training_data.csv", "a+") as f:
        f.write("\n".join(data.to_csv(None, index=False).split("\n")[1:]))
        f.write("\n")

Palette images with Transparency expressed in bytes should be converted to RGBA images


In [ ]:
training_data = pd.read_csv("training_data.csv")
training_data["label"] = y_train
training_data.to_csv("training_data.csv", index=False)

In [12]:
with open("validation_data.csv", "w") as f:
    f.write(",".join(map(str, range(512))))
    f.write("\n")

for i in range(0, len(valid_images), 200):
    data = preprocess_images(valid_images[i:i+200])
    with open("validation_data.csv", "a+") as f:
        f.write("\n".join(data.to_csv(None, index=False).split("\n")[1:]))
        f.write("\n")

Palette images with Transparency expressed in bytes should be converted to RGBA images


In [13]:
validation_data = pd.read_csv("validation_data.csv")
validation_data["label"] = y_valid
validation_data.to_csv("validation_data.csv", index=False)

In [14]:
with open("test_data.csv", "w") as f:
    f.write(",".join(map(str, range(512))))
    f.write("\n")

for i in range(0, len(test_images), 200):
    data = preprocess_images(test_images[i:i+200])
    with open("test_data.csv", "a+") as f:
        f.write("\n".join(data.to_csv(None, index=False).split("\n")[1:]))
        f.write("\n")

In [15]:
test_data = pd.read_csv("test_data.csv")
test_data["label"] = y_test
test_data.to_csv("test_data.csv", index=False)

## Load data for training

In [2]:
training_data = pd.read_csv("training_data.csv")
validation_data = pd.read_csv("validation_data.csv")
test_data = pd.read_csv("test_data.csv")

In [3]:
print(training_data.shape, validation_data.shape, test_data.shape)
training_data.head()

(75369, 513) (15376, 513) (15018, 513)


,0,1,2,3,4,5,6,7,8,9,...,503,504,505,506,507,508,509,510,511,label
0,-0.036284,0.065120,-0.026674,0.037901,-0.023275,0.023836,0.002879,0.007602,0.011652,-0.048367,...,-0.037090,0.030496,0.035004,0.056767,0.044238,-0.025507,-0.046314,0.003570,-0.024306,non-nazi
1,-0.015738,0.093096,0.084752,0.112216,-0.000442,-0.065532,0.007091,-0.025539,-0.045250,0.040717,...,-0.064796,-0.022393,0.011908,-0.008220,-0.018699,0.013882,0.019758,-0.008252,-0.018672,non-nazi
2,0.018969,0.124155,-0.094642,-0.000600,-0.057754,-0.037017,0.009728,-0.046778,0.006461,-0.032716,...,0.084787,0.011991,-0.006625,-0.057902,-0.011498,0.018157,-0.002892,0.011497,0.022816,non-nazi
3,-0.004954,-0.224413,-0.049862,-0.022130,-0.010407,-0.041640,-0.035328,0.006612,-0.002796,0.007758,...,0.023775,-0.027235,-0.019263,0.046171,0.055021,-0.029527,0.041510,-0.026703,-0.059962,non-nazi
4,-0.013319,-0.039820,-0.079986,0.040090,-0.033282,0.037759,0.001027,0.018503,-0.020362,-0.111674,...,0.060417,-0.009748,-0.003844,-0.061079,0.012986,0.034320,-0.022465,-0.052805,0.005296,non-nazi


In [4]:
feature_columns = [str(i) for i in range(512)]
label_column = "label"

In [5]:
test_features = test_data[feature_columns]
test_labels = test_data[label_column].tolist()

In [6]:
y_true = [int(label == "nazi-symbol") for label in test_labels]

## Train and evaluate multiple classifiers

In [7]:
predicted_result = dict()

def gather_result(classifier):
    print("score on test: " + str(classifier.score(test_features, test_labels)))
    if getattr(classifier, "predict_proba", None):
        scores = pd.DataFrame(data=classifier.predict_proba(test_features), columns=classifier.classes_)
        outputs = [int(prob > 0.5) for prob in scores["nazi-symbol"]]
        fpr, tpr, roc_threshold = roc_curve(y_true, scores["nazi-symbol"])
        roc_auc = auc(fpr, tpr)
        precision, recall, pr_threshold = precision_recall_curve(y_true, scores["nazi-symbol"])
        average_precision = average_precision_score(y_true, scores["nazi-symbol"])
        predicted_result[type(classifier).__name__] = dict(precision=precision, recall=recall, pr_threshold=pr_threshold, 
                                                           average_precision=average_precision, fpr=fpr, tpr=tpr, 
                                                           roc_threshold=roc_threshold, roc_auc=roc_auc,
                                                           y_true=y_true, outputs=outputs, probs = scores["nazi-symbol"])
    else:
        outputs = [int(label == "nazi-symbol") for label in classifier.predict(test_features)]
        predicted_result[type(classifier).__name__] = dict(y_true=y_true, outputs=outputs)
    print(classification_report(y_true, outputs, digits=3))
    print("accuracy score:", accuracy_score(y_true, outputs))
    if getattr(classifier, "predict_proba", None): 
        print("roc auc score:", roc_auc_score(y_true, scores["nazi-symbol"]))

In [31]:
%%time

# import the library
from sklearn.linear_model import LogisticRegression

# instantiate & fit
lr=LogisticRegression(max_iter=5000)
lr.fit(training_data[feature_columns], training_data[label_column])

CPU times: user 6.96 s, sys: 1.2 s, total: 8.16 s
Wall time: 455 ms


LogisticRegression(max_iter=5000)

In [32]:
gather_result(lr)

score on test: 0.9970701824477294
              precision    recall  f1-score   support

           0      0.998     0.999     0.999     14813
           1      0.909     0.873     0.891       205

    accuracy                          0.997     15018
   macro avg      0.953     0.936     0.945     15018
weighted avg      0.997     0.997     0.997     15018

accuracy score: 0.9970701824477294
roc auc score: 0.9995346869015845


In [33]:
%%time

# import the library
from sklearn.linear_model import SGDClassifier

# instantiate & fit
sgd=SGDClassifier()
sgd.fit(training_data[feature_columns], training_data[label_column])

CPU times: user 342 ms, sys: 78.2 ms, total: 420 ms
Wall time: 418 ms


SGDClassifier()

In [34]:
gather_result(sgd)

score on test: 0.9970701824477294
              precision    recall  f1-score   support

           0      0.998     0.999     0.999     14813
           1      0.897     0.888     0.892       205

    accuracy                          0.997     15018
   macro avg      0.947     0.943     0.945     15018
weighted avg      0.997     0.997     0.997     15018

accuracy score: 0.9970701824477294


In [35]:
%%time

# import the library
from sklearn.neighbors import KNeighborsClassifier

# instantiate & fit
knn = KNeighborsClassifier(algorithm = 'brute', n_jobs=-1)
knn.fit(training_data[feature_columns], training_data[label_column])

CPU times: user 122 ms, sys: 64.1 ms, total: 186 ms
Wall time: 184 ms


KNeighborsClassifier(algorithm='brute', n_jobs=-1)

In [36]:
gather_result(knn)

score on test: 0.9977360500732454
              precision    recall  f1-score   support

           0      0.999     0.998     0.999     14813
           1      0.887     0.956     0.920       205

    accuracy                          0.998     15018
   macro avg      0.943     0.977     0.960     15018
weighted avg      0.998     0.998     0.998     15018

accuracy score: 0.9977360500732454
roc auc score: 0.9994693191379358


In [8]:
%%time

# import the library
from sklearn.svm import SVC

# instantiate & fit
svm=SVC(C=3)
svm.fit(training_data[feature_columns], training_data[label_column])

CPU times: user 1min 37s, sys: 101 ms, total: 1min 37s
Wall time: 1min 37s


SVC(C=3)

In [9]:
gather_result(svm)
import joblib
joblib.dump(svm, "first-layer.pt")

score on test: 0.9985350912238647
              precision    recall  f1-score   support

           0      0.999     0.999     0.999     14813
           1      0.946     0.946     0.946       205

    accuracy                          0.999     15018
   macro avg      0.973     0.973     0.973     15018
weighted avg      0.999     0.999     0.999     15018

accuracy score: 0.9985350912238647


['first-layer.pt']

In [39]:
%%time

# import the library
from sklearn.tree import DecisionTreeClassifier

# instantiate & fit
clf = DecisionTreeClassifier(min_samples_split=10,max_depth=3)
clf.fit(training_data[feature_columns], training_data[label_column])

CPU times: user 14.8 s, sys: 43.8 ms, total: 14.9 s
Wall time: 14.9 s


DecisionTreeClassifier(max_depth=3, min_samples_split=10)

In [40]:
gather_result(clf)

score on test: 0.9898788120921561
              precision    recall  f1-score   support

           0      0.997     0.993     0.995     14813
           1      0.602     0.761     0.672       205

    accuracy                          0.990     15018
   macro avg      0.799     0.877     0.834     15018
weighted avg      0.991     0.990     0.990     15018

accuracy score: 0.9898788120921561
roc auc score: 0.9332957372644002


In [41]:
%%time

# import the library
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

# instantiate & fit
bg=BaggingClassifier(DecisionTreeClassifier(min_samples_split=10,max_depth=3),max_samples=0.5,max_features=1.0,n_estimators=10)
bg.fit(training_data[feature_columns], training_data[label_column])

CPU times: user 59.2 s, sys: 202 ms, total: 59.4 s
Wall time: 59.4 s


BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=3,
                                                   min_samples_split=10),
                  max_samples=0.5)

In [42]:
gather_result(bg)

score on test: 0.9910107870555334
              precision    recall  f1-score   support

           0      0.996     0.995     0.995     14813
           1      0.651     0.737     0.691       205

    accuracy                          0.991     15018
   macro avg      0.824     0.866     0.843     15018
weighted avg      0.992     0.991     0.991     15018

accuracy score: 0.9910107870555334
roc auc score: 0.9641175434234596


In [43]:
%%time

# import the library
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

# instantiate & fit
adb = AdaBoostClassifier(DecisionTreeClassifier(max_depth=2),n_estimators=100,learning_rate=0.5)
adb.fit(training_data[feature_columns], training_data[label_column])

CPU times: user 17min 14s, sys: 2.97 s, total: 17min 17s
Wall time: 17min 17s


AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=2),
                   learning_rate=0.5, n_estimators=100)

In [44]:
gather_result(adb)

score on test: 0.9963377280596617
              precision    recall  f1-score   support

           0      0.998     0.999     0.998     14813
           1      0.899     0.824     0.860       205

    accuracy                          0.996     15018
   macro avg      0.948     0.912     0.929     15018
weighted avg      0.996     0.996     0.996     15018

accuracy score: 0.9963377280596617
roc auc score: 0.9981275511128161


In [45]:
%%time

# import the library
from sklearn.ensemble import GradientBoostingClassifier

# instantiate & fit
gbc = GradientBoostingClassifier(n_estimators=100)
gbc.fit(training_data[feature_columns], training_data[label_column])

CPU times: user 24min 14s, sys: 74.9 ms, total: 24min 14s
Wall time: 24min 15s


GradientBoostingClassifier()

In [46]:
gather_result(gbc)

score on test: 0.9959382074843521
              precision    recall  f1-score   support

           0      0.998     0.998     0.998     14813
           1      0.875     0.820     0.846       205

    accuracy                          0.996     15018
   macro avg      0.936     0.909     0.922     15018
weighted avg      0.996     0.996     0.996     15018

accuracy score: 0.9959382074843521
roc auc score: 0.9982572987142144


In [47]:
%%time

# import the library
from sklearn.ensemble import RandomForestClassifier

# instantiate & fit
rf = RandomForestClassifier(n_estimators=300,max_depth=3)
rf.fit(training_data[feature_columns], training_data[label_column])

CPU times: user 2min 2s, sys: 52 ms, total: 2min 2s
Wall time: 2min 2s


RandomForestClassifier(max_depth=3, n_estimators=300)

In [48]:
gather_result(rf)

score on test: 0.9876814489279532
              precision    recall  f1-score   support

           0      0.988     1.000     0.994     14813
           1      1.000     0.098     0.178       205

    accuracy                          0.988     15018
   macro avg      0.994     0.549     0.586     15018
weighted avg      0.988     0.988     0.983     15018

accuracy score: 0.9876814489279532
roc auc score: 0.9984762889551531


In [49]:
%%time

# import the library
from sklearn.ensemble import VotingClassifier
from sklearn.svm import SVC

evc=VotingClassifier(estimators=[('lr', LogisticRegression(max_iter=5000)),
                                 ('rf', RandomForestClassifier(n_estimators=30,max_depth=3)),
                                 ('svm', SVC(max_iter=5000))])
evc.fit(training_data[feature_columns], training_data[label_column])

CPU times: user 1min 27s, sys: 3.61 s, total: 1min 31s
Wall time: 1min 21s


VotingClassifier(estimators=[('lr', LogisticRegression(max_iter=5000)),
                             ('rf',
                              RandomForestClassifier(max_depth=3,
                                                     n_estimators=30)),
                             ('svm', SVC(max_iter=5000))])

In [50]:
gather_result(evc)

score on test: 0.9974031162604874
              precision    recall  f1-score   support

           0      0.998     0.999     0.999     14813
           1      0.932     0.873     0.902       205

    accuracy                          0.997     15018
   macro avg      0.965     0.936     0.950     15018
weighted avg      0.997     0.997     0.997     15018

accuracy score: 0.9974031162604874


In [51]:
joblib.dump(predicted_result, 'openclip-encoder-output/openclip-encoder-predicted-results.joblib')

['openclip-encoder-output/openclip-encoder-predicted-results.joblib']

In [29]:
predicted_result

{'LogisticRegression': {'precision': array([0.01365029, 0.0136512 , 0.0136521 , ..., 1.        , 1.        ,
         1.        ]),
  'recall': array([1.        , 1.        , 1.        , ..., 0.0097561 , 0.00487805,
         0.        ]),
  'pr_threshold': array([1.89827946e-06, 2.16975849e-06, 2.22459708e-06, ...,
         9.96336889e-01, 9.97712356e-01, 9.98449383e-01]),
  'average_precision': 0.9639975831276607,
  'fpr': array([0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 6.75082698e-05,
         6.75082698e-05, 1.35016540e-04, 1.35016540e-04, 2.02524809e-04,
         2.02524809e-04, 2.70033079e-04, 2.70033079e-04, 4.05049619e-04,
         4.05049619e-04, 4.72557888e-04, 4.72557888e-04, 5.40066158e-04,
         5.40066158e-04, 6.07574428e-04, 6.07574428e-04, 6.75082698e-04,
         6.75082698e-04, 8.77607507e-04, 8.77607507e-04, 9.45115777e-04,
         9.45115777e-04, 1.01262405e-03, 1.01262405e-03, 1.08013232e-03,
         1.08013232e-03, 1.14764059e-03, 1.14764059e-03, 1.2151